In [ ]:
from flask import (
    Flask,
    request,
    jsonify,
    redirect,
    render_template,
    flash
)

import sqlite3
import string
import random
import re
from datetime import datetime
from urllib.parse import urlparse


app = Flask(__name__)

app.secret_key = "change-this-secret-key"

DATABASE = "urls.db"


# ============================================================
# DATABASE
# ============================================================

def get_db():
    conn = sqlite3.connect(DATABASE)
    conn.row_factory = sqlite3.Row
    return conn


def init_db():

    conn = get_db()

    conn.execute("""
        CREATE TABLE IF NOT EXISTS urls (

            id INTEGER PRIMARY KEY AUTOINCREMENT,

            original_url TEXT NOT NULL,

            short_code TEXT UNIQUE NOT NULL,

            clicks INTEGER DEFAULT 0,

            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP

        )
    """)

    conn.commit()

    conn.close()


# ============================================================
# URL VALIDATION
# ============================================================

def is_valid_url(url):

    try:

        result = urlparse(url)

        return (
            result.scheme in ["http", "https"]
            and result.netloc != ""
        )

    except Exception:

        return False


# ============================================================
# SHORT CODE GENERATOR
# ============================================================

def generate_short_code(length=6):

    characters = (
        string.ascii_letters +
        string.digits
    )

    while True:

        short_code = "".join(
            random.choices(
                characters,
                k=length
            )
        )

        conn = get_db()

        existing = conn.execute(
            """
            SELECT id
            FROM urls
            WHERE short_code = ?
            """,
            (short_code,)
        ).fetchone()

        conn.close()

        if not existing:

            return short_code


# ============================================================
# HOME PAGE
# ============================================================

@app.route("/")
def index():

    conn = get_db()

    urls = conn.execute("""
        SELECT *
        FROM urls
        ORDER BY created_at DESC
    """).fetchall()

    conn.close()

    return render_template(
        "index.html",
        urls=urls
    )


# ============================================================
# CREATE SHORT URL - WEB
# ============================================================

@app.route("/shorten", methods=["POST"])
def shorten():

    original_url = request.form.get(
        "url",
        ""
    ).strip()

    # Validation

    if not original_url:

        flash(
            "Please enter a URL.",
            "error"
        )

        return redirect("/")


    if not is_valid_url(original_url):

        flash(
            "Please enter a valid URL starting with http:// or https://",
            "error"
        )

        return redirect("/")


    # Generate unique code

    short_code = generate_short_code()


    # Store in database

    conn = get_db()

    conn.execute(
        """
        INSERT INTO urls
        (
            original_url,
            short_code
        )
        VALUES (?, ?)
        """,
        (
            original_url,
            short_code
        )
    )

    conn.commit()

    conn.close()


    flash(
        f"Short URL created: /{short_code}",
        "success"
    )

    return redirect("/")


# ============================================================
# REDIRECT SHORT URL
# ============================================================

@app.route("/<short_code>")
def redirect_to_url(short_code):

    conn = get_db()

    url = conn.execute(
        """
        SELECT *
        FROM urls
        WHERE short_code = ?
        """,
        (short_code,)
    ).fetchone()


    if not url:

        conn.close()

        return "Short URL not found.", 404


    # Increment click counter

    conn.execute(
        """
        UPDATE urls

        SET clicks = clicks + 1

        WHERE short_code = ?
        """,
        (short_code,)
    )

    conn.commit()

    conn.close()


    # HTTP redirect

    return redirect(
        url["original_url"],
        code=302
    )


# ============================================================
# DELETE URL
# ============================================================

@app.route(
    "/delete/<short_code>",
    methods=["POST"]
)
def delete_url(short_code):

    conn = get_db()

    url = conn.execute(
        """
        SELECT *
        FROM urls
        WHERE short_code = ?
        """,
        (short_code,)
    ).fetchone()


    if not url:

        conn.close()

        flash(
            "URL not found.",
            "error"
        )

        return redirect("/")


    conn.execute(
        """
        DELETE FROM urls
        WHERE short_code = ?
        """,
        (short_code,)
    )

    conn.commit()

    conn.close()


    flash(
        "Short URL deleted successfully.",
        "success"
    )

    return redirect("/")


# ============================================================
# REST API - CREATE SHORT URL
# ============================================================

@app.route(
    "/api/shorten",
    methods=["POST"]
)
def api_shorten():

    data = request.get_json(
        silent=True
    )


    if not data:

        return jsonify({
            "success": False,
            "error": "JSON request body required."
        }), 400


    original_url = data.get(
        "url",
        ""
    ).strip()


    # Validate URL

    if not original_url:

        return jsonify({
            "success": False,
            "error": "URL is required."
        }), 400


    if not is_valid_url(original_url):

        return jsonify({
            "success": False,
            "error": "Invalid URL."
        }), 400


    # Generate code

    short_code = generate_short_code()


    # Insert database record

    conn = get_db()

    cursor = conn.execute(
        """
        INSERT INTO urls
        (
            original_url,
            short_code
        )
        VALUES (?, ?)
        """,
        (
            original_url,
            short_code
        )
    )

    conn.commit()


    url_id = cursor.lastrowid

    conn.close()


    return jsonify({

        "success": True,

        "id": url_id,

        "original_url": original_url,

        "short_code": short_code,

        "short_url": request.host_url + short_code

    }), 201


# ============================================================
# REST API - GET URL
# ============================================================

@app.route(
    "/api/url/<short_code>",
    methods=["GET"]
)
def api_get_url(short_code):

    conn = get_db()

    url = conn.execute(
        """
        SELECT *
        FROM urls
        WHERE short_code = ?
        """,
        (short_code,)
    ).fetchone()

    conn.close()


    if not url:

        return jsonify({

            "success": False,

            "error": "URL not found."

        }), 404


    return jsonify({

        "success": True,

        "id": url["id"],

        "original_url": url["original_url"],

        "short_code": url["short_code"],

        "clicks": url["clicks"],

        "created_at": url["created_at"],

        "short_url":
            request.host_url +
            url["short_code"]

    })


# ============================================================
# REST API - GET ALL URLS
# ============================================================

@app.route(
    "/api/urls",
    methods=["GET"]
)
def api_get_all_urls():

    conn = get_db()

    urls = conn.execute(
        """
        SELECT *
        FROM urls
        ORDER BY created_at DESC
        """
    ).fetchall()

    conn.close()


    result = []


    for url in urls:

        result.append({

            "id": url["id"],

            "original_url":
                url["original_url"],

            "short_code":
                url["short_code"],

            "clicks":
                url["clicks"],

            "created_at":
                url["created_at"],

            "short_url":
                request.host_url +
                url["short_code"]

        })


    return jsonify({

        "success": True,

        "count": len(result),

        "urls": result

    })


# ============================================================
# REST API - DELETE URL
# ============================================================

@app.route(
    "/api/url/<short_code>",
    methods=["DELETE"]
)
def api_delete_url(short_code):

    conn = get_db()

    url = conn.execute(
        """
        SELECT *
        FROM urls
        WHERE short_code = ?
        """,
        (short_code,)
    ).fetchone()


    if not url:

        conn.close()

        return jsonify({

            "success": False,

            "error": "URL not found."

        }), 404


    conn.execute(
        """
        DELETE FROM urls
        WHERE short_code = ?
        """,
        (short_code,)
    )

    conn.commit()

    conn.close()


    return jsonify({

        "success": True,

        "message":
            "Short URL deleted successfully."

    })


# ============================================================
# RUN APPLICATION
# ============================================================

if __name__ == "__main__":

    init_db()

    app.run(
        debug=True,
        host="127.0.0.1",
        port=5000
    )

In [ ]:
<!DOCTYPE html>

<html lang="en">

<head>

    <meta charset="UTF-8">

    <meta
        name="viewport"
        content="width=device-width, initial-scale=1.0"
    >

    <title>URL Shortener</title>

    <link
        rel="stylesheet"
        href="{{ url_for('static',
                         filename='style.css') }}"
    >

</head>


<body>


<div class="container">

    <div class="header">

        <h1>URL Shortener</h1>

        <p>
            Convert long URLs into short,
            shareable links.
        </p>

    </div>


    <!-- FLASH MESSAGES -->

    {% with messages =
           get_flashed_messages(
               with_categories=true
           )
    %}

        {% if messages %}

            {% for category, message in messages %}

                <div class="alert {{ category }}">

                    {{ message }}

                </div>

            {% endfor %}

        {% endif %}

    {% endwith %}


    <!-- URL FORM -->

    <div class="card">

        <h2>Create Short URL</h2>

        <form
            method="POST"
            action="{{ url_for('shorten') }}"
        >

            <input
                type="url"
                name="url"
                placeholder="https://example.com/very-long-url"
                required
            >

            <button type="submit">
                Shorten URL
            </button>

        </form>

    </div>


    <!-- URL LIST -->

    <div class="card">

        <h2>Shortened URLs</h2>


        {% if urls %}

        <div class="table-container">

            <table>

                <thead>

                    <tr>

                        <th>Original URL</th>

                        <th>Short Link</th>

                        <th>Clicks</th>

                        <th>Created</th>

                        <th>Action</th>

                    </tr>

                </thead>


                <tbody>

                {% for url in urls %}

                    <tr>

                        <td class="original-url">

                            {{ url["original_url"] }}

                        </td>


                        <td>

                            <a
                                href="{{ request.host_url }}{{ url['short_code'] }}"
                                target="_blank"
                            >

                                {{ request.host_url }}{{ url["short_code"] }}

                            </a>

                        </td>


                        <td>

                            <span class="click-count">

                                {{ url["clicks"] }}

                            </span>

                        </td>


                        <td>

                            {{ url["created_at"] }}

                        </td>


                        <td>

                            <form
                                method="POST"
                                action="{{ url_for(
                                    'delete_url',
                                    short_code=url['short_code']
                                ) }}"
                                onsubmit="return confirmDelete();"
                            >

                                <button
                                    type="submit"
                                    class="delete-btn"
                                >

                                    Delete

                                </button>

                            </form>

                        </td>

                    </tr>

                {% endfor %}

                </tbody>

            </table>

        </div>

        {% else %}

            <div class="empty">

                <p>
                    No shortened URLs yet.
                </p>

            </div>

        {% endif %}

    </div>


    <!-- API INFORMATION -->

    <div class="card api-card">

        <h2>REST API</h2>

        <p>
            This application also provides
            REST API endpoints.
        </p>


        <div class="endpoint">

            <strong>POST</strong>

            <code>/api/shorten</code>

            <span>
                Create a shortened URL
            </span>

        </div>


        <div class="endpoint">

            <strong>GET</strong>

            <code>/api/url/&lt;short_code&gt;</code>

            <span>
                Retrieve URL information
            </span>

        </div>


        <div class="endpoint">

            <strong>GET</strong>

            <code>/api/urls</code>

            <span>
                Retrieve all URLs
            </span>

        </div>


        <div class="endpoint">

            <strong>DELETE</strong>

            <code>/api/url/&lt;short_code&gt;</code>

            <span>
                Delete a shortened URL
            </span>

        </div>

    </div>

</div>


<script
    src="{{ url_for('static',
                    filename='script.js') }}"
></script>


</body>

</html>

In [ ]:
* {
    box-sizing: border-box;
    margin: 0;
    padding: 0;
}


body {

    font-family:
        Arial,
        Helvetica,
        sans-serif;

    background: #f4f6f8;

    color: #222;

    min-height: 100vh;

}


.container {

    width: 90%;

    max-width: 1200px;

    margin: 50px auto;

}


.header {

    text-align: center;

    margin-bottom: 35px;

}


.header h1 {

    font-size: 40px;

    margin-bottom: 10px;

}


.header p {

    color: #666;

    font-size: 17px;

}


.card {

    background: white;

    padding: 30px;

    margin-bottom: 25px;

    border-radius: 10px;

    box-shadow:
        0 3px 15px
        rgba(0, 0, 0, 0.08);

}


.card h2 {

    margin-bottom: 20px;

}


form {

    display: flex;

    gap: 10px;

}


input[type="url"] {

    flex: 1;

    padding: 14px;

    border: 1px solid #ccc;

    border-radius: 6px;

    font-size: 16px;

}


button {

    padding: 14px 22px;

    border: none;

    border-radius: 6px;

    background: #222;

    color: white;

    cursor: pointer;

    font-size: 15px;

}


button:hover {

    opacity: 0.85;

}


.table-container {

    overflow-x: auto;

}


table {

    width: 100%;

    border-collapse: collapse;

}


th,
td {

    padding: 14px;

    border-bottom: 1px solid #eee;

    text-align: left;

    vertical-align: middle;

}


th {

    background: #f7f7f7;

}


td a {

    color: #2563eb;

    text-decoration: none;

}


td a:hover {

    text-decoration: underline;

}


.original-url {

    max-width: 300px;

    word-break: break-all;

}


.click-count {

    display: inline-block;

    min-width: 30px;

    padding: 5px 9px;

    background: #eee;

    border-radius: 15px;

    text-align: center;

}


.delete-btn {

    background: #dc3545;

    padding: 8px 12px;

}


.alert {

    padding: 15px;

    margin-bottom: 20px;

    border-radius: 6px;

}


.alert.success {

    background: #d4edda;

    color: #155724;

}


.alert.error {

    background: #f8d7da;

    color: #721c24;

}


.empty {

    text-align: center;

    padding: 40px;

    color: #777;

}


.api-card {

    margin-top: 25px;

}


.api-card p {

    color: #666;

    margin-bottom: 20px;

}


.endpoint {

    display: flex;

    align-items: center;

    gap: 15px;

    padding: 12px;

    background: #f7f7f7;

    margin-bottom: 10px;

    border-radius: 5px;

}


.endpoint strong {

    min-width: 60px;

}


.endpoint code {

    font-family: monospace;

}


.endpoint span {

    color: #666;

}


@media (max-width: 700px) {

    .container {

        width: 94%;

        margin: 25px auto;

    }


    .header h1 {

        font-size: 30px;

    }


    .card {

        padding: 20px;

    }


    form {

        flex-direction: column;

    }


    .endpoint {

        flex-direction: column;

        align-items: flex-start;

        gap: 5px;

    }

}

In [ ]:
function confirmDelete() {

    return confirm(
        "Are you sure you want to delete this shortened URL?"
    );

}


setTimeout(function () {

    const alerts =
        document.querySelectorAll(".alert");

    alerts.forEach(function (alert) {

        alert.style.opacity = "0";

        alert.style.transition =
            "opacity 0.5s";

        setTimeout(function () {

            alert.remove();

        }, 500);

    });

}, 4000);